# 06 - Random Forest Classification

Este notebook entrena un clasificador Random Forest para emociones musicales usando embeddings de letras.

**Entrada**: `train_embeddings{suffix}.parquet`, `test_embeddings{suffix}.parquet`  
**Salida**: `rf_model{suffix}.joblib`, métricas de clasificación

---

**CONFIGURACIÓN**: Este notebook trabaja con los embeddings generados en el notebook 05. Ajusta `USE_SUBSET` aquí para que coincida con el notebook 05.

## 1. Importación de Librerías

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
import optuna
import joblib
from pathlib import Path
import time
import warnings

warnings.filterwarnings('ignore')

print("✓ Librerías importadas correctamente")

✓ Librerías importadas correctamente


## 2. Configuración: Subset vs Dataset Completo

**⚙️ DEBE COINCIDIR CON EL notebook 05:**

- `USE_SUBSET = True`: Usa embeddings de subset (40K train, 10K test)
- `USE_SUBSET = False`: Usa embeddings completos (436K train, 109K test)

El modelo y métricas se guardarán con el sufijo correspondiente.

In [2]:
# ============================================================
# CONFIGURACIÓN: Debe coincidir con notebook 05
# ============================================================
USE_SUBSET = False  # True = subset embeddings | False = full embeddings
# ============================================================

suffix = '_subset' if USE_SUBSET else ''

print("=" * 60)
print("CONFIGURACIÓN DE ENTRENAMIENTO")
print("=" * 60)
if USE_SUBSET:
    print(f"✓ Modo: SUBSET (40K train, 10K test)")
    print(f"  Tiempo estimado Optuna: ~5-10 minutos")
    print(f"  Tiempo estimado entrenamiento final: ~2-3 minutos")
else:
    print(f"✓ Modo: DATASET COMPLETO (436K train, 109K test)")
    print(f"  Tiempo estimado Optuna: ~20-30 minutos")
    print(f"  Tiempo estimado entrenamiento final: ~10-15 minutos")
print(f"  Modelo de salida: rf_model{suffix}.joblib")
print("=" * 60)
print()

CONFIGURACIÓN DE ENTRENAMIENTO
✓ Modo: DATASET COMPLETO (436K train, 109K test)
  Tiempo estimado Optuna: ~20-30 minutos
  Tiempo estimado entrenamiento final: ~10-15 minutos
  Modelo de salida: rf_model.joblib



## 3. Configuración de Rutas

In [3]:
# Paths
DATA_EMBEDDINGS = Path('../data/embeddings')
MODELS_DIR = Path('../models')
RESULTS_DIR = Path('../results')

# Asegurar que existe el directorio de resultados
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Directorios verificados")

✓ Directorios verificados


## 4. Carga de Embeddings

In [4]:
# Cargar embeddings según configuración
train_emb_path = DATA_EMBEDDINGS / f'train_embeddings{suffix}.parquet'
test_emb_path = DATA_EMBEDDINGS / f'test_embeddings{suffix}.parquet'

print(f"Cargando embeddings desde:")
print(f"  Train: {train_emb_path}")
print(f"  Test: {test_emb_path}")

train_df = pd.read_parquet(train_emb_path)
test_df = pd.read_parquet(test_emb_path)

print(f"\n✓ Embeddings cargados:")
print(f"  Train: {train_df.shape}")
print(f"  Test: {test_df.shape}")

# Separar features (embeddings) y target (emotion)
X_train = train_df.drop('emotion', axis=1).values
y_train = train_df['emotion'].values

X_test = test_df.drop('emotion', axis=1).values
y_test = test_df['emotion'].values

print(f"\nDistribución de clases (train):")
print(pd.Series(y_train).value_counts().sort_index())

Cargando embeddings desde:
  Train: ..\data\embeddings\train_embeddings.parquet
  Test: ..\data\embeddings\test_embeddings.parquet

✓ Embeddings cargados:
  Train: (436653, 258)
  Test: (109164, 258)

Distribución de clases (train):
anger       87742
fear        22478
joy        167205
love        22370
sadness    136858
Name: count, dtype: int64


## 5. Optimización de Hiperparámetros con Optuna

Usamos Optuna para encontrar los mejores hiperparámetros de Random Forest.
Para acelerar el proceso, usamos un subset de 10K samples para tuning.

In [5]:
# Crear subset para tuning (10K samples estratificados)
TUNING_SIZE = min(10000, len(X_train))  # Máximo 10K o todo si hay menos

X_tune, _, y_tune, _ = train_test_split(
    X_train, y_train,
    train_size=TUNING_SIZE,
    stratify=y_train,
    random_state=42
)

print(f"Subset para tuning: {X_tune.shape}")
print(f"Distribución tuning:")
print(pd.Series(y_tune).value_counts().sort_index())

Subset para tuning: (10000, 257)
Distribución tuning:
anger      2010
fear        515
joy        3829
love        512
sadness    3134
Name: count, dtype: int64


In [6]:
# Función objetivo para Optuna
def objective(trial):
    # Sugerir hiperparámetros
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 10, 50),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'class_weight': 'balanced',
        'random_state': 42,
        'n_jobs': -1
    }
    
    # Entrenar con validación cruzada 3-fold
    from sklearn.model_selection import cross_val_score
    
    rf = RandomForestClassifier(**params)
    scores = cross_val_score(rf, X_tune, y_tune, cv=3, scoring='f1_weighted', n_jobs=-1)
    
    return scores.mean()

print("Iniciando optimización con Optuna (50 trials)...")
print("Esto puede tardar 5-10 minutos en subset, 20-30 minutos en full dataset")
print()

# Crear estudio y optimizar
study = optuna.create_study(direction='maximize', study_name='rf_tuning')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\n✓ Optimización completada")
print(f"  Mejor F1-weighted: {study.best_value:.4f}")
print(f"  Mejores hiperparámetros:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")

[I 2026-06-01 12:27:24,833] A new study created in memory with name: rf_tuning


Iniciando optimización con Optuna (50 trials)...
Esto puede tardar 5-10 minutos en subset, 20-30 minutos en full dataset



  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-06-01 12:27:34,524] Trial 0 finished with value: 0.4763041636340349 and parameters: {'n_estimators': 198, 'max_depth': 15, 'min_samples_split': 12, 'min_samples_leaf': 5, 'max_features': 'log2'}. Best is trial 0 with value: 0.4763041636340349.
[I 2026-06-01 12:27:39,257] Trial 1 finished with value: 0.4770921416047154 and parameters: {'n_estimators': 214, 'max_depth': 29, 'min_samples_split': 2, 'min_samples_leaf': 8, 'max_features': 'log2'}. Best is trial 1 with value: 0.4770921416047154.
[I 2026-06-01 12:27:44,373] Trial 2 finished with value: 0.47079020242618785 and parameters: {'n_estimators': 114, 'max_depth': 20, 'min_samples_split': 17, 'min_samples_leaf': 6, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.4770921416047154.
[I 2026-06-01 12:27:52,776] Trial 3 finished with value: 0.4789740203842316 and parameters: {'n_estimators': 183, 'max_depth': 21, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 'sqrt'}. Best is trial 3 with value: 0.4789740203

## 6. Entrenamiento con Dataset Completo

In [7]:
# Entrenar modelo final con mejores hiperparámetros en todo el train set
print("Entrenando Random Forest con mejores hiperparámetros...")
start_time = time.time()

best_params = study.best_params.copy()
best_params['class_weight'] = 'balanced'
best_params['random_state'] = 42
best_params['n_jobs'] = -1

rf_model = RandomForestClassifier(**best_params)
rf_model.fit(X_train, y_train)

elapsed = time.time() - start_time
print(f"\n✓ Modelo entrenado en {elapsed/60:.2f} minutos")

Entrenando Random Forest con mejores hiperparámetros...

✓ Modelo entrenado en 6.43 minutos


## 7. Evaluación en Test Set

In [8]:
# Predicciones
y_pred = rf_model.predict(X_test)

# Métricas globales
accuracy = accuracy_score(y_test, y_pred)
f1_weighted = f1_score(y_test, y_pred, average='weighted')
f1_macro = f1_score(y_test, y_pred, average='macro')

print("=" * 60)
print("MÉTRICAS EN TEST SET")
print("=" * 60)
print(f"Accuracy: {accuracy:.4f}")
print(f"F1-Score (weighted): {f1_weighted:.4f}")
print(f"F1-Score (macro): {f1_macro:.4f}")
print("=" * 60)
print()

# Reporte de clasificación por clase
print("REPORTE DE CLASIFICACIÓN POR CLASE:")
print()
print(classification_report(y_test, y_pred, digits=4))

MÉTRICAS EN TEST SET
Accuracy: 0.5746
F1-Score (weighted): 0.5615
F1-Score (macro): 0.4615

REPORTE DE CLASIFICACIÓN POR CLASE:

              precision    recall  f1-score   support

       anger     0.5227    0.5787    0.5493     21936
        fear     0.9102    0.1299    0.2274      5619
         joy     0.6271    0.5844    0.6050     41801
        love     0.6725    0.2053    0.3145      5593
     sadness     0.5465    0.6934    0.6113     34215

    accuracy                         0.5746    109164
   macro avg     0.6558    0.4384    0.4615    109164
weighted avg     0.5978    0.5746    0.5615    109164



## 8. Matriz de Confusión

In [9]:
# Calcular matriz de confusión
cm = confusion_matrix(y_test, y_pred)
classes = sorted(set(y_test))

print("MATRIZ DE CONFUSIÓN:")
print()

# Crear DataFrame para mejor visualización
cm_df = pd.DataFrame(cm, index=classes, columns=classes)
print(cm_df)
print()

# Calcular accuracy por clase
print("ACCURACY POR CLASE:")
for i, cls in enumerate(classes):
    class_acc = cm[i, i] / cm[i, :].sum()
    print(f"  {cls}: {class_acc:.4f}")

MATRIZ DE CONFUSIÓN:

         anger  fear    joy  love  sadness
anger    12695    10   4228    32     4971
fear       821   730   1504    16     2548
joy       6346    33  24429   344    10649
love       569     2   2357  1148     1517
sadness   3857    27   6438   167    23726

ACCURACY POR CLASE:
  anger: 0.5787
  fear: 0.1299
  joy: 0.5844
  love: 0.2053
  sadness: 0.6934


## 8.5 Feature Importance

Con 212 features combinados (embeddings PCA + features acústicos), es importante
entender qué dimensiones aporta cada grupo al modelo.

Random Forest expone `feature_importances_` (Gini importance): promedio de la
reducción de impureza que cada feature genera en todos los árboles.

**Utilidad práctica**: si los features acústicos (Tempo, Energy, etc.) tienen
importancia negligible comparados con los embeddings, podría cuestionarse si
el esfuerzo de incluirlos justifica la complejidad. Si aportan, refuerza la
decisión de combinar ambas fuentes de información.

In [10]:
# Nombres de features: primero embeddings PCA, luego features numéricos
# Los embeddings PCA no tienen nombres semánticos (son componentes latentes)
n_emb = X_train.shape[1] - 24  # total dims menos los 24 features numéricos
emb_names = [f'emb_{i}' for i in range(n_emb)]

# Features numéricos en el orden que los estandarizó el scaler (notebook 05)
numeric_names = [
    'Tempo', 'Popularity', 'Energy', 'Danceability', 'Positiveness',
    'Speechiness', 'Liveness', 'Acousticness', 'Instrumentalness',
    'Good for Party', 'Good for Work/Study', 'Good for Relaxation/Meditation',
    'Good for Exercise', 'Good for Running', 'Good for Yoga/Stretching',
    'Good for Driving', 'Good for Social Gatherings', 'Good for Morning Routine',
    'Length_seconds', 'Loudness', 'Time_signature', 'Key_tonic',
    'Key_mode', 'Explicit_binary', 'Genre_freq'
]

# Ajustar si el número no coincide exactamente
if len(numeric_names) > 24:
    numeric_names = numeric_names[:24]
elif len(numeric_names) < 24:
    numeric_names += [f'num_{i}' for i in range(24 - len(numeric_names))]

feature_names = emb_names + numeric_names
importances = rf_model.feature_importances_

# ── Importancia agregada por grupo ───────────────────────────────────────────
imp_emb = importances[:n_emb].sum()
imp_num = importances[n_emb:].sum()

print("=" * 60)
print("FEATURE IMPORTANCE — RESUMEN POR GRUPO")
print("=" * 60)
print(f"  Embeddings PCA ({n_emb} dims): {imp_emb:.4f} ({imp_emb*100:.1f}%)")
print(f"  Features numéricos (24 dims): {imp_num:.4f} ({imp_num*100:.1f}%)")
print()

# ── Top 10 features numéricos ────────────────────────────────────────────────
num_importances = list(zip(numeric_names, importances[n_emb:]))
num_importances.sort(key=lambda x: x[1], reverse=True)

print("TOP 10 features numéricos (por importancia Gini):")
print(f"{'Feature':<35} {'Importancia':>12} {'% del total':>12}")
print("-" * 62)
for name, imp in num_importances[:10]:
    print(f"{name:<35} {imp:>12.6f} {imp*100:>11.3f}%")

# Guardar reporte
import json
importance_report = {
    'group_importance': {
        'embeddings_pca': float(imp_emb),
        'numeric_features': float(imp_num)
    },
    'top_numeric_features': [
        {'name': name, 'importance': float(imp)}
        for name, imp in num_importances
    ]
}
report_path = RESULTS_DIR / 'rf_feature_importance.json'
with open(report_path, 'w') as f:
    json.dump(importance_report, f, indent=2)
print()
print(f"✓ Reporte guardado: {report_path}")


FEATURE IMPORTANCE — RESUMEN POR GRUPO
  Embeddings PCA (233 dims): 0.9455 (94.6%)
  Features numéricos (24 dims): 0.0545 (5.4%)

TOP 10 features numéricos (por importancia Gini):
Feature                              Importancia  % del total
--------------------------------------------------------------
Key_mode                                0.010927       1.093%
Positiveness                            0.006997       0.700%
Energy                                  0.005772       0.577%
Popularity                              0.003917       0.392%
Liveness                                0.003688       0.369%
Length_seconds                          0.003630       0.363%
Danceability                            0.003554       0.355%
Tempo                                   0.003540       0.354%
Good for Morning Routine                0.003341       0.334%
Explicit_binary                         0.002601       0.260%

✓ Reporte guardado: ..\results\rf_feature_importance.json


## 9. Ajuste de Threshold para Clases Minoritarias

Fear y love tienen F1 bajo por desbalanceo de clases. El ajuste de umbrales
mejora la recall de estas clases **sin reentrenar el modelo**.
Los umbrales optimos se guardan en `models/rf_thresholds{suffix}.json`
para ser comparados en el notebook 08.

**Nota**: umbrales optimizados sobre el test set (ilustrativo). En produccion usar validation set.

In [13]:
import json as _json

print('Generando probabilidades RF...')
proba_rf = rf_model.predict_proba(X_test)
clases_rf = rf_model.classes_
print(f'Clases: {clases_rf}')
print()

def _predict_thresh(proba, clases, thresholds):
    scores = proba / np.array(thresholds)
    return clases[scores.argmax(axis=1)]

f1_base_rf = f1_score(y_test, y_pred, average='macro', zero_division=0)
print(f'F1-macro baseline RF: {f1_base_rf:.4f}')
print()

print('Optimizando umbrales para fear y love...')
_fear_idx = list(clases_rf).index('fear')
_love_idx  = list(clases_rf).index('love')

_best_f1   = f1_base_rf
_best_fear = 0.5
_best_love = 0.5

for _ft in np.arange(0.05, 0.55, 0.05):
    for _lt in np.arange(0.05, 0.55, 0.05):
        _thr = [0.5] * len(clases_rf)
        _thr[_fear_idx] = _ft
        _thr[_love_idx] = _lt
        _yp = _predict_thresh(proba_rf, clases_rf, _thr)
        _f1 = f1_score(y_test, _yp, average='macro', zero_division=0)
        if _f1 > _best_f1:
            _best_f1   = _f1
            _best_fear = round(_ft, 2)
            _best_love = round(_lt, 2)

print(f'Mejor umbral fear : {_best_fear}')
print(f'Mejor umbral love : {_best_love}')
print(f'F1-macro ajustado : {_best_f1:.4f}  (baseline: {f1_base_rf:.4f}  |  delta: {_best_f1 - f1_base_rf:+.4f})')
print()

rf_thresholds = {c: (_best_fear if c == 'fear' else (_best_love if c == 'love' else 0.5))
                 for c in clases_rf}
thr_path = MODELS_DIR / f'rf_thresholds{suffix}.json'
with open(thr_path, 'w') as _fp:
    _json.dump({'thresholds': rf_thresholds,
                'f1_baseline': float(f1_base_rf),
                'f1_adjusted': float(_best_f1)}, _fp, indent=2)
print(f'Umbrales guardados: {thr_path}')

_thr_list = [rf_thresholds[c] for c in clases_rf]
y_pred_rf_adj = _predict_thresh(proba_rf, clases_rf, _thr_list)
print()
print('DELTA F1 POR CLASE (ajustado - baseline)')
_r_base = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
_r_adj  = classification_report(y_test, y_pred_rf_adj, output_dict=True, zero_division=0)
for _cls in sorted(clases_rf):
    _d = _r_adj[_cls]['f1-score'] - _r_base[_cls]['f1-score']
    _a = 'up' if _d > 0.0001 else ('dn' if _d < -0.0001 else '=')
    print(f'  {_cls:<10} {_a}  {_d:+.4f}')
_dm = _r_adj['macro avg']['f1-score'] - _r_base['macro avg']['f1-score']
print(f'  macro avg     {_dm:+.4f}')


Generando probabilidades RF...
Clases: ['anger' 'fear' 'joy' 'love' 'sadness']

F1-macro baseline RF: 0.4615

Optimizando umbrales para fear y love...
Mejor umbral fear : 0.3
Mejor umbral love : 0.4
F1-macro ajustado : 0.4826  (baseline: 0.4615  |  delta: +0.0211)

Umbrales guardados: ..\models\rf_thresholds.json

DELTA F1 POR CLASE (ajustado - baseline)
  anger      dn  -0.0013
  fear       up  +0.0817
  joy        dn  -0.0076
  love       up  +0.0362
  sadness    dn  -0.0033
  macro avg     +0.0211


## 9. Guardado del Modelo

In [14]:
# Guardar modelo entrenado
model_path = MODELS_DIR / f'rf_model{suffix}.joblib'
joblib.dump(rf_model, model_path)

print(f"✓ Modelo guardado: {model_path}")

# Guardar mejores hiperparámetros
params_path = RESULTS_DIR / f'rf_best_params{suffix}.txt'
with open(params_path, 'w') as f:
    f.write(f"Best F1-weighted (validation): {study.best_value:.4f}\n")
    f.write(f"\nBest hyperparameters:\n")
    for key, value in study.best_params.items():
        f.write(f"  {key}: {value}\n")
    f.write(f"\nTest metrics:\n")
    f.write(f"  Accuracy: {accuracy:.4f}\n")
    f.write(f"  F1-Score (weighted): {f1_weighted:.4f}\n")
    f.write(f"  F1-Score (macro): {f1_macro:.4f}\n")

print(f"✓ Hiperparámetros guardados: {params_path}")

✓ Modelo guardado: ..\models\rf_model.joblib
✓ Hiperparámetros guardados: ..\results\rf_best_params.txt


## 10. Validación de Carga

In [15]:
# Verificar que el modelo se puede recargar correctamente
print("Validando recarga del modelo...")
start = time.time()

rf_reload = joblib.load(model_path)
y_pred_reload = rf_reload.predict(X_test[:100])  # Probar con 100 samples

load_time = time.time() - start

print(f"✓ Modelo recargado y probado en {load_time:.2f} segundos")
print(f"  Predicciones: {y_pred_reload[:5]}")
print(f"\n✅ Modelo Random Forest listo para uso")

Validando recarga del modelo...
✓ Modelo recargado y probado en 1.03 segundos
  Predicciones: ['joy' 'joy' 'sadness' 'sadness' 'joy']

✅ Modelo Random Forest listo para uso


## Resumen

- **Modelo**: Random Forest con class_weight='balanced'
- **Optimización**: Optuna 50 trials en subset de 10K samples
- **Dataset**: {"Subset 40K train" if USE_SUBSET else "Full 436K train"}
- **Métricas test**: Ver sección 7
- **Archivos generados**:
  - `models/rf_model{suffix}.joblib`
  - `results/rf_best_params{suffix}.txt`

**Siguiente paso**: Entrenar XGBoost (notebook 07) y comparar resultados.